# Ch 30. ZeROとFSDP — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: 70Bモデル、N=16についてZeRO Stage 0/1/2/3のメモリを計算せよ。

### 解法

混合精度Adamの単純modelはFP16 parameter 2P、gradient 2P、FP32 master+moment 12Pで計16P byte。Stage 1/2/3はoptimizer、gradient、parameterまで順に$N$分割する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
P=70; N=16
mem={'stage0':16*P,'stage1':4*P+12*P/N,'stage2':4*P+(2+12)*P/N,'stage3':16*P/N}
assert mem['stage3']==70.; mem


## 問題 2

**問題**: FSDPが順伝播前にAll-Gatherを行う理由を説明せよ。

### 解法

各rankはparameter shardのみ保持するがdense layer計算には全weightが必要。順伝播直前にAll-Gatherで一時再構成し使用後reshardする。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
P,N=1000,4; local=P//N; gathered=local*N
assert gathered==P
{'local_shard':local,'required_for_layer_compute':gathered}


## 問題 3

**問題**: ZeRO-3の通信オーバーヘッドをDDPと比較せよ。

### 解法

DDP ring all-reduceはgradient約$2(N-1)P/N$を通信する。ZeRO-3はforward/backwardのparameter all-gatherもありeventが増えるためbucket/prefetch overlapが重要。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
P=1.; ddp=2*(1-1/8)*P; zero3_forward=P; zero3_backward=2*P
assert zero3_forward+zero3_backward>ddp
{'DDP_ring_units':ddp,'ZeRO3_approx_units':zero3_forward+zero3_backward}


## 問題 4

**問題**: CPU Offloadingが実行速度を低下させる理由を説明せよ。

### 解法

offloadはGPU HBMでなく低速CPU DRAMにstateを置きPCIe/NVLink転送をcritical pathへ加える。転送下限はbytes/bandwidthで、小kernelと同期がlatencyを増す。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
bytes_to_move=8e9; pcie_bandwidth=32e9; transfer_seconds=bytes_to_move/pcie_bandwidth
assert transfer_seconds>.2; {'one_way_transfer_lower_bound_seconds':transfer_seconds}


## 問題 5

**問題**: 70Bモデルを学習するためZeRO-3とA100 80GBが何台必要か計算せよ。

### 解法

70Bの16P stateは約1120 GB。80 GBの15%をactivation・fragmentation用に残すと有効68 GBで、理想下限は$\lceil1120/68\rceil=17$台。実数はcheckpointingとsequence長で検証する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import math
parameters_billion=70; bytes_per_parameter=16; gpu_capacity_gb=80; headroom=.15
total_gb=parameters_billion*bytes_per_parameter
usable_per_gpu=gpu_capacity_gb*(1-headroom)
minimum=math.ceil(total_gb/usable_per_gpu)
feasible=lambda n: n*usable_per_gpu>=total_gb
enumerated=next(n for n in range(1,1000) if feasible(n))
assert minimum==enumerated and feasible(minimum) and not feasible(minimum-1)
{'model_state_gb':total_gb,'usable_gb_per_gpu':usable_per_gpu,'ideal_minimum':minimum,
 'boundary_check':{'previous_capacity_gb':(minimum-1)*usable_per_gpu,'minimum_capacity_gb':minimum*usable_per_gpu},
 'note':'ideal model-state bound only; measured activations and fragmentation can require more GPUs'}
